# Standalone forecast models

This notebook copies **CNN-LSTM, N-BEATS, Transformer, SARIMA, Holt-Winters**, and an **inverse-MAE ensemble** into cells.

It does **not** import `forecasting`, `main`, or any other project `.py` file.

Allowed libraries: `numpy`, `pandas`, `scikit-learn`, `statsmodels`, `matplotlib` (optional `tensorflow` for Keras CNN-LSTM).

Run **all cells in order**. Deep-learning demos use a small epoch count so they finish on CPU.


## 1. Third-party imports only


In [ ]:
from __future__ import annotations

import itertools
from pathlib import Path

import numpy as np
import pandas as pd
from sklearn.preprocessing import MinMaxScaler

try:
    import statsmodels.api as sm
    from statsmodels.tsa.holtwinters import ExponentialSmoothing
except ImportError as exc:
    raise ImportError("Install statsmodels to run SARIMA and Holt-Winters.") from exc

try:
    import matplotlib.pyplot as plt
    HAS_MPL = True
except ImportError:
    HAS_MPL = False

SEED = 42
LOOKBACK = 12
HORIZON = 6
DEMO_EPOCHS = 8
np.random.seed(SEED)
print("Ready. No project modules imported.")


## 2. Shared NumPy utilities (activations, dropout, Adam, early stop)


In [ ]:
def tqdm(iterable=None, **kwargs):
    return iterable if iterable is not None else range(kwargs.get("total", 0))


# ---------------------------------------------------------------------------
# Activation functions
# ---------------------------------------------------------------------------

def relu(x: np.ndarray) -> np.ndarray:
    """Rectified Linear Unit."""
    return np.maximum(0, x)


def sigmoid(x: np.ndarray) -> np.ndarray:
    """Numerically stable sigmoid."""
    return 1.0 / (1.0 + np.exp(-np.clip(x, -30, 30)))


def tanh(x: np.ndarray) -> np.ndarray:
    """Clipped hyperbolic tangent."""
    return np.tanh(np.clip(x, -30, 30))


def epoch_bar(n_epochs: int, model_name: str):
    """tqdm progress bar over training epochs (visible in terminal + Gradio)."""
    return tqdm(
        range(n_epochs),
        desc=f"{model_name} epochs",
        unit="ep",
        leave=True,
        dynamic_ncols=True,
    )


def apply_dropout(x: np.ndarray, rate: float, training: bool) -> np.ndarray:
    """Inverted dropout — active only when *training* is True."""
    if (not training) or rate <= 0.0:
        return x
    keep = 1.0 - float(rate)
    mask = (np.random.rand(*x.shape) < keep).astype(x.dtype)
    return x * mask / keep


class EarlyStopTracker:
    """
    Stop training when epoch loss stops decreasing.

    If loss does not improve by *min_delta* for *patience* consecutive
    epochs, ``step`` returns True and the caller should restore
    ``best_state`` then break out of the epoch loop.
    """

    def __init__(self, patience: int = 5, min_delta: float = 1e-6) -> None:
        self.patience = max(1, int(patience))
        self.min_delta = float(min_delta)
        self.best_loss = np.inf
        self.bad_epochs = 0
        self.best_state = None

    def step(self, loss: float, state: dict | None = None) -> bool:
        """
        Record *loss* for this epoch.

        Returns
        -------
        bool
            True → no improvement for *patience* epochs (stop training).
        """
        if loss < self.best_loss - self.min_delta:
            self.best_loss = float(loss)
            self.bad_epochs = 0
            if state is not None:
                self.best_state = {k: np.copy(v) for k, v in state.items()}
            return False
        self.bad_epochs += 1
        return self.bad_epochs >= self.patience


# ---------------------------------------------------------------------------
# Adam optimiser
# ---------------------------------------------------------------------------

class AdamOptimizer:
    """
    Minimal, per-key stateful Adam optimiser (Kingma & Ba, 2015).

    Each parameter tensor is identified by a string *key*; state (m, v)
    is stored in dictionaries so the same instance can manage multiple
    independent parameter tensors.

    Parameters
    ----------
    lr  : float  Learning rate (α).
    b1  : float  Exponential decay rate for 1st-moment estimate.
    b2  : float  Exponential decay rate for 2nd-moment estimate.
    eps : float  Numerical stability constant.
    """

    def __init__(self, lr: float = 1e-3, b1: float = 0.9,
                 b2: float = 0.999, eps: float = 1e-8) -> None:
        self.lr, self.b1, self.b2, self.eps = lr, b1, b2, eps
        self.t: int = 0
        self.m: dict = {}
        self.v: dict = {}

    def update(self, key: str, param: np.ndarray,
               grad: np.ndarray) -> np.ndarray:
        """
        Compute and return the updated parameter tensor.

        Parameters
        ----------
        key   : str        Unique name identifying this parameter.
        param : np.ndarray Current parameter value.
        grad  : np.ndarray Gradient w.r.t. *param*.

        Returns
        -------
        np.ndarray  Updated parameter.
        """
        if key not in self.m:
            self.m[key] = np.zeros_like(param)
            self.v[key] = np.zeros_like(param)

        self.t     += 1
        self.m[key] = self.b1 * self.m[key] + (1 - self.b1) * grad
        self.v[key] = self.b2 * self.v[key] + (1 - self.b2) * grad ** 2

        m_hat = self.m[key] / (1 - self.b1 ** self.t)
        v_hat = self.v[key] / (1 - self.b2 ** self.t)

        return param - self.lr * m_hat / (np.sqrt(v_hat) + self.eps)


## 3. CNN-LSTM (inlined)


In [ ]:
class CnnLstmForecaster:
    """
    1-D CNN feature extractor stacked before an LSTM, producing a
    multi-step forecast via a single Dense output layer.

    Architecture
    ------------
    Input  : (W, F)  — lookback window of F features
    Conv1  : kernel=3, 16 filters  → ReLU → Dropout
    Conv2  : kernel=3,  8 filters  → ReLU → Dropout
    LSTM   : hidden size 32 → Dropout
    Dense  : 32 → horizon

    Training uses analytic gradients for the Dense layer and sparse
    numerical gradients (128 random elements per step) for the LSTM
    weights. Epochs stop early when loss stops decreasing.

    Parameters
    ----------
    lookback       : int   Number of historical time steps in the input window.
    n_features     : int   Number of features per time step.
    horizon        : int   Number of future steps to forecast.
    lr             : float Adam learning rate.
    epochs         : int   Max training epochs.
    seed           : int   NumPy random seed for reproducibility.
    dropout        : float Dropout rate applied after Conv / LSTM (train only).
    early_patience : int   Stop after this many epochs with no loss decrease.
    """

    def __init__(self, lookback: int = 12, n_features: int = 1,
                 horizon: int = 12, lr: float = 1e-2,
                 epochs: int = 80, seed: int = 42,
                 dropout: float = 0.2, early_patience: int = 5) -> None:
        np.random.seed(seed)
        self.W      = lookback
        self.F      = n_features
        self.H      = horizon
        self.lr     = lr
        self.epochs = epochs
        self.dropout = float(dropout)
        self.early_patience = int(early_patience)
        self.H_lstm = 32
        self._init_weights()

    # ------------------------------------------------------------------
    # Weight initialisation
    # ------------------------------------------------------------------

    def _init_weights(self) -> None:
        # Conv1: kernel=3, F_in=F, F_out=16
        self.Wc1 = np.random.randn(3, self.F, 16) * 0.1
        self.bc1 = np.zeros(16)
        # Conv2: kernel=3, F_in=16, F_out=8
        self.Wc2 = np.random.randn(3, 16, 8) * 0.1
        self.bc2 = np.zeros(8)
        # LSTM: input_size=8, hidden=32
        H, L_in    = self.H_lstm, 8
        self.Wlstm = np.random.randn(L_in + H, 4 * H) * 0.05
        self.blstm = np.zeros(4 * H)
        self.blstm[H:2 * H] = 1.0   # forget-gate bias initialised to 1
        # Dense: H -> horizon
        self.Wd = np.random.randn(H, self.H) * 0.1
        self.bd = np.zeros(self.H)

    def _snapshot(self) -> dict:
        return {
            "Wc1": self.Wc1, "bc1": self.bc1,
            "Wc2": self.Wc2, "bc2": self.bc2,
            "Wlstm": self.Wlstm, "blstm": self.blstm,
            "Wd": self.Wd, "bd": self.bd,
        }

    def _restore(self, state: dict) -> None:
        for k, v in state.items():
            setattr(self, k, np.copy(v))

    # ------------------------------------------------------------------
    # Forward primitives
    # ------------------------------------------------------------------

    def _conv1d(self, x: np.ndarray, W: np.ndarray,
                b: np.ndarray) -> np.ndarray:
        """Valid-padded 1-D convolution."""
        k, _, C_out = W.shape
        T           = x.shape[0]
        out         = np.zeros((T - k + 1, C_out))
        for i in range(T - k + 1):
            out[i] = x[i:i + k].reshape(-1) @ W.reshape(-1, C_out) + b
        return out

    def _lstm_step(self, x_seq: np.ndarray) -> np.ndarray:
        """Run LSTM over *x_seq* and return final hidden state."""
        H   = self.H_lstm
        h   = np.zeros(H)
        c   = np.zeros(H)
        for xt in x_seq:
            combined = np.concatenate([xt, h])
            gates    = combined @ self.Wlstm + self.blstm
            ig = sigmoid(gates[:H])
            fg = sigmoid(gates[H:2 * H])
            g  = tanh   (gates[2 * H:3 * H])
            og = sigmoid(gates[3 * H:])
            c  = fg * c + ig * g
            h  = og * tanh(c)
        return h

    def _forward(self, x: np.ndarray, training: bool = False) -> np.ndarray:
        c1 = relu(self._conv1d(x, self.Wc1, self.bc1))
        c1 = apply_dropout(c1, self.dropout, training)
        c2 = relu(self._conv1d(c1, self.Wc2, self.bc2))
        c2 = apply_dropout(c2, self.dropout, training)
        h  = self._lstm_step(c2)
        h  = apply_dropout(h, self.dropout, training)
        return h @ self.Wd + self.bd

    # ------------------------------------------------------------------
    # Training
    # ------------------------------------------------------------------

    def fit(self, X: np.ndarray, y: np.ndarray) -> "CnnLstmForecaster":
        """
        Train on sliding-window dataset.

        Parameters
        ----------
        X : np.ndarray  Shape (N, W, F).
        y : np.ndarray  Shape (N, horizon).
        """
        opt = AdamOptimizer(self.lr)
        N   = X.shape[0]
        stopper = EarlyStopTracker(patience=self.early_patience)
        bar = epoch_bar(self.epochs, "CNN-LSTM")

        for _ in bar:
            epoch_loss = 0.0
            for i in np.random.permutation(N):
                xi, yi = X[i], y[i]
                pred   = self._forward(xi, training=True)
                epoch_loss += float(np.mean((pred - yi) ** 2))
                dL     = 2 * (pred - yi) / self.H

                # Analytic Dense gradient (eval path, no dropout noise)
                c1  = relu(self._conv1d(xi, self.Wc1, self.bc1))
                c2  = relu(self._conv1d(c1, self.Wc2, self.bc2))
                h   = self._lstm_step(c2)
                self.Wd = opt.update("Wd", self.Wd, np.outer(h, dL))
                self.bd = opt.update("bd", self.bd, dL)

                # Sparse numerical gradient for LSTM parameters
                eps = 1e-3
                for name in ["Wlstm", "blstm"]:
                    W_    = getattr(self, name)
                    grad  = np.zeros_like(W_)
                    n_upd = min(W_.size, 128)
                    idxs  = np.random.choice(W_.size, n_upd, replace=False)
                    flat  = W_.ravel()
                    for j in idxs:
                        orig    = flat[j]
                        flat[j] = orig + eps
                        setattr(self, name, flat.reshape(W_.shape))
                        pp = self._forward(xi, training=False)
                        flat[j] = orig - eps
                        setattr(self, name, flat.reshape(W_.shape))
                        pm = self._forward(xi, training=False)
                        flat[j] = orig
                        setattr(self, name, flat.reshape(W_.shape))
                        grad.ravel()[j] = (
                            np.mean((pp - yi) ** 2) - np.mean((pm - yi) ** 2)
                        ) / (2 * eps)
                    setattr(self, name,
                            opt.update(name, getattr(self, name), grad))

            mean_loss = epoch_loss / max(N, 1)
            if hasattr(bar, "set_postfix"):
                bar.set_postfix(loss=f"{mean_loss:.4f}")
            if stopper.step(mean_loss, self._snapshot()):
                if stopper.best_state is not None:
                    self._restore(stopper.best_state)
                break
        else:
            if stopper.best_state is not None:
                self._restore(stopper.best_state)
        return self

    # ------------------------------------------------------------------
    # Inference
    # ------------------------------------------------------------------

    def predict(self, X: np.ndarray) -> np.ndarray:
        """
        Parameters
        ----------
        X : np.ndarray  Shape (N, W, F).

        Returns
        -------
        np.ndarray  Shape (N, horizon).
        """
        return np.array([self._forward(xi, training=False) for xi in X])


## 4. N-BEATS (inlined)


In [ ]:
# ---------------------------------------------------------------------------
# N-BEATS building block
# ---------------------------------------------------------------------------

class NBeatsBlock:
    """
    Single N-BEATS block: 3-layer fully-connected stack producing
    backcast and forecast via polynomial basis projections.

    Parameters
    ----------
    in_dim    : int   Input dimensionality (lookback * n_features).
    theta_dim : int   Polynomial basis degree.
    H         : int   Forecast horizon.
    hidden    : int   Width of FC hidden layers.
    seed      : int   Random seed.
    dropout   : float Dropout rate on FC hidden activations (train only).
    """

    def __init__(self, in_dim: int, theta_dim: int, H: int,
                 hidden: int = 64, seed: int = 0,
                 dropout: float = 0.2) -> None:
        np.random.seed(seed)
        self.dropout = float(dropout)
        # FC stack
        self.W1  = np.random.randn(in_dim, hidden) * 0.05
        self.b1  = np.zeros(hidden)
        self.W2  = np.random.randn(hidden, hidden) * 0.05
        self.b2  = np.zeros(hidden)
        self.W3  = np.random.randn(hidden, hidden) * 0.05
        self.b3  = np.zeros(hidden)
        # Basis projection heads
        self.Wtb = np.random.randn(hidden, theta_dim) * 0.05
        self.Wtf = np.random.randn(hidden, theta_dim) * 0.05
        self.btb = np.zeros(theta_dim)
        self.btf = np.zeros(theta_dim)
        # Vandermonde basis matrices
        t_b      = np.linspace(-1,  0, in_dim)
        t_f      = np.linspace( 0,  1, H)
        self.Vb  = np.vstack([t_b ** k for k in range(theta_dim)]).T
        self.Vf  = np.vstack([t_f ** k for k in range(theta_dim)]).T

    def forward(self, x: np.ndarray, training: bool = False):
        """
        Parameters
        ----------
        x : np.ndarray  Shape (in_dim,).

        Returns
        -------
        tuple[np.ndarray, np.ndarray]
            ``(backcast, forecast)`` both 1-D.
        """
        h  = relu(x  @ self.W1 + self.b1)
        h  = apply_dropout(h, self.dropout, training)
        h  = relu(h  @ self.W2 + self.b2)
        h  = apply_dropout(h, self.dropout, training)
        h  = relu(h  @ self.W3 + self.b3)
        h  = apply_dropout(h, self.dropout, training)
        bc = (h @ self.Wtb + self.btb) @ self.Vb.T
        fc = (h @ self.Wtf + self.btf) @ self.Vf.T
        return bc, fc


# ---------------------------------------------------------------------------
# N-BEATS stack forecaster
# ---------------------------------------------------------------------------

class NBeatsForecaster:
    """
    N-BEATS forecaster: 2 stacks × 3 blocks each.

    Training uses sparse numerical gradients on the forecast-head weights
    (``Wtf``, ``btf``) only. Epochs stop early when loss stops decreasing.

    Parameters
    ----------
    lookback       : int   Lookback window length.
    n_features     : int   Features per time step.
    horizon        : int   Forecast horizon.
    lr             : float Adam learning rate.
    epochs         : int   Max training epochs.
    seed           : int   Random seed.
    dropout        : float Dropout rate on FC hidden layers.
    early_patience : int   Stop after this many epochs with no loss decrease.
    """

    def __init__(self, lookback: int = 12, n_features: int = 1,
                 horizon: int = 12, lr: float = 5e-3,
                 epochs: int = 100, seed: int = 42,
                 dropout: float = 0.2, early_patience: int = 5) -> None:
        np.random.seed(seed)
        self.W       = lookback * n_features
        self.H       = horizon
        self.lr      = lr
        self.epochs  = epochs
        self.dropout = float(dropout)
        self.early_patience = int(early_patience)
        self.blocks  = [
            NBeatsBlock(
                self.W, 8, horizon, hidden=64,
                seed=s * 10 + b, dropout=self.dropout,
            )
            for s in range(2) for b in range(3)
        ]

    def _forward(self, x: np.ndarray, training: bool = False) -> np.ndarray:
        residual = x.copy()
        forecast = np.zeros(self.H)
        for blk in self.blocks:
            bc, fc   = blk.forward(residual, training=training)
            residual  = residual - bc
            forecast  = forecast + fc
        return forecast

    def _snapshot(self) -> dict:
        state = {}
        for i, blk in enumerate(self.blocks):
            for attr in ("W1", "b1", "W2", "b2", "W3", "b3",
                         "Wtb", "Wtf", "btb", "btf"):
                state[f"b{i}_{attr}"] = getattr(blk, attr)
        return state

    def _restore(self, state: dict) -> None:
        for i, blk in enumerate(self.blocks):
            for attr in ("W1", "b1", "W2", "b2", "W3", "b3",
                         "Wtb", "Wtf", "btb", "btf"):
                key = f"b{i}_{attr}"
                if key in state:
                    setattr(blk, attr, np.copy(state[key]))

    def fit(self, X: np.ndarray, y: np.ndarray) -> "NBeatsForecaster":
        """
        Parameters
        ----------
        X : np.ndarray  Shape (N, W, F).
        y : np.ndarray  Shape (N, horizon).
        """
        N   = X.shape[0]
        Xf  = X.reshape(N, -1)
        opt = AdamOptimizer(self.lr)
        eps = 1e-3
        stopper = EarlyStopTracker(patience=self.early_patience)
        bar = epoch_bar(self.epochs, "N-BEATS")

        for _ in bar:
            epoch_loss = 0.0
            for i in np.random.permutation(N):
                xi, yi = Xf[i], y[i]
                pred = self._forward(xi, training=True)
                epoch_loss += float(np.mean((pred - yi) ** 2))
                for bi, blk in enumerate(self.blocks):
                    for attr in ["Wtf", "btf"]:
                        W_   = getattr(blk, attr)
                        grad = np.zeros_like(W_)
                        n_up = min(W_.size, 40)
                        idxs = np.random.choice(W_.size, n_up, replace=False)
                        flat = W_.ravel()
                        for j in idxs:
                            orig    = flat[j]
                            flat[j] = orig + eps
                            setattr(blk, attr, flat.reshape(W_.shape))
                            pp = self._forward(xi, training=False)
                            flat[j] = orig - eps
                            setattr(blk, attr, flat.reshape(W_.shape))
                            pm = self._forward(xi, training=False)
                            flat[j] = orig
                            setattr(blk, attr, flat.reshape(W_.shape))
                            grad.ravel()[j] = (
                                np.mean((pp - yi) ** 2) - np.mean((pm - yi) ** 2)
                            ) / (2 * eps)
                        setattr(blk, attr,
                                opt.update(f"b{bi}_{attr}",
                                           getattr(blk, attr), grad))

            mean_loss = epoch_loss / max(N, 1)
            if hasattr(bar, "set_postfix"):
                bar.set_postfix(loss=f"{mean_loss:.4f}")
            if stopper.step(mean_loss, self._snapshot()):
                if stopper.best_state is not None:
                    self._restore(stopper.best_state)
                break
        else:
            if stopper.best_state is not None:
                self._restore(stopper.best_state)
        return self

    def predict(self, X: np.ndarray) -> np.ndarray:
        """
        Parameters
        ----------
        X : np.ndarray  Shape (N, W, F).

        Returns
        -------
        np.ndarray  Shape (N, horizon).
        """
        Xf = X.reshape(X.shape[0], -1)
        return np.array([self._forward(xi, training=False) for xi in Xf])


## 5. Transformer (inlined)


In [ ]:
class TransformerForecaster:
    """
    Lightweight Transformer encoder: positional embedding → 2 × multi-head
    self-attention + FFN → mean pool → Dense forecast.

    Training uses analytic gradients for the Dense layer and sparse
    numerical gradients (50 random elements per step) for key matrices.
    Epochs stop early when loss stops decreasing; dropout regularises
    attention / FFN activations during training.

    Parameters
    ----------
    lookback       : int   Lookback window length.
    n_features     : int   Features per time step.
    horizon        : int   Forecast horizon.
    d_model        : int   Embedding / model dimension.
    n_heads        : int   Number of attention heads (d_model must be divisible).
    lr             : float Adam learning rate.
    epochs         : int   Max training epochs.
    seed           : int   Random seed.
    dropout        : float Dropout rate (train only).
    early_patience : int   Stop after this many epochs with no loss decrease.
    """

    def __init__(self, lookback: int = 12, n_features: int = 1,
                 horizon: int = 12, d_model: int = 16, n_heads: int = 2,
                 lr: float = 3e-3, epochs: int = 80,
                 seed: int = 42, dropout: float = 0.2,
                 early_patience: int = 5) -> None:
        np.random.seed(seed)
        self.W  = lookback
        self.F  = n_features
        self.H  = horizon
        self.dm = d_model
        self.nh = n_heads
        self.dh = d_model // n_heads
        self.lr = lr
        self.epochs = epochs
        self.dropout = float(dropout)
        self.early_patience = int(early_patience)
        self._init_weights()

    # ------------------------------------------------------------------
    # Weight initialisation
    # ------------------------------------------------------------------

    def _init_weights(self) -> None:
        dm = self.dm

        # Input projection + sinusoidal positional encoding
        self.We = np.random.randn(self.F, dm) * 0.1
        self.be = np.zeros(dm)

        pos       = np.arange(self.W)[:, None]
        div       = np.exp(np.arange(0, dm, 2) * (-np.log(10_000) / dm))
        self.PE   = np.zeros((self.W, dm))
        self.PE[:, 0::2] = np.sin(pos * div)
        cols_cos  = np.arange(1, dm, 2)
        self.PE[:, cols_cos] = np.cos(pos * div[:len(cols_cos)])

        # Multi-head attention weights
        self.Wq = np.random.randn(self.nh, dm, self.dh) * 0.05
        self.Wk = np.random.randn(self.nh, dm, self.dh) * 0.05
        self.Wv = np.random.randn(self.nh, dm, self.dh) * 0.05
        self.Wo = np.random.randn(self.nh * self.dh, dm) * 0.05

        # Feed-forward network
        self.Wff1 = np.random.randn(dm, 32) * 0.05
        self.bff1 = np.zeros(32)
        self.Wff2 = np.random.randn(32, dm) * 0.05
        self.bff2 = np.zeros(dm)

        # Output Dense
        self.Wd = np.random.randn(dm, self.H) * 0.1
        self.bd = np.zeros(self.H)

    def _snapshot(self) -> dict:
        return {
            "We": self.We, "be": self.be,
            "Wq": self.Wq, "Wk": self.Wk, "Wv": self.Wv, "Wo": self.Wo,
            "Wff1": self.Wff1, "bff1": self.bff1,
            "Wff2": self.Wff2, "bff2": self.bff2,
            "Wd": self.Wd, "bd": self.bd,
        }

    def _restore(self, state: dict) -> None:
        for k, v in state.items():
            setattr(self, k, np.copy(v))

    # ------------------------------------------------------------------
    # Forward primitives
    # ------------------------------------------------------------------

    def _attn(self, x: np.ndarray) -> np.ndarray:
        """Multi-head scaled dot-product self-attention."""
        heads = []
        for h in range(self.nh):
            Q  = x @ self.Wq[h]
            K  = x @ self.Wk[h]
            V  = x @ self.Wv[h]
            sc = Q @ K.T / np.sqrt(self.dh)
            sc -= sc.max(axis=-1, keepdims=True)
            A  = np.exp(sc) / (np.exp(sc).sum(axis=-1, keepdims=True) + 1e-9)
            heads.append(A @ V)
        return np.concatenate(heads, axis=-1) @ self.Wo

    def _forward(self, x: np.ndarray, training: bool = False) -> np.ndarray:
        """
        Parameters
        ----------
        x : np.ndarray  Shape (W, F).

        Returns
        -------
        np.ndarray  Shape (horizon,).
        """
        e  = x @ self.We + self.be + self.PE
        e  = e + apply_dropout(self._attn(e), self.dropout, training)
        ff = relu(e @ self.Wff1 + self.bff1)
        ff = apply_dropout(ff, self.dropout, training)
        ff = ff @ self.Wff2 + self.bff2
        e  = e + apply_dropout(ff, self.dropout, training)
        e  = e + apply_dropout(self._attn(e), self.dropout, training)
        return e.mean(axis=0) @ self.Wd + self.bd

    # ------------------------------------------------------------------
    # Training
    # ------------------------------------------------------------------

    def fit(self, X: np.ndarray, y: np.ndarray) -> "TransformerForecaster":
        """
        Parameters
        ----------
        X : np.ndarray  Shape (N, W, F).
        y : np.ndarray  Shape (N, horizon).
        """
        opt = AdamOptimizer(self.lr)
        N   = X.shape[0]
        eps = 1e-3
        stopper = EarlyStopTracker(patience=self.early_patience)
        bar = epoch_bar(self.epochs, "Transformer")

        for _ in bar:
            epoch_loss = 0.0
            for i in np.random.permutation(N):
                xi, yi = X[i], y[i]
                pred   = self._forward(xi, training=True)
                epoch_loss += float(np.mean((pred - yi) ** 2))
                dL     = 2 * (pred - yi) / self.H

                # Analytic Dense gradient (approximate via mean-pool)
                pooled  = (xi @ self.We + self.be + self.PE).mean(axis=0)
                self.Wd = opt.update("Wd", self.Wd, np.outer(pooled, dL))
                self.bd = opt.update("bd", self.bd, dL)

                # Sparse numerical gradients for key matrices
                for name in ["Wo", "Wff2", "We"]:
                    W_   = getattr(self, name)
                    grad = np.zeros_like(W_)
                    n_up = min(W_.size, 50)
                    idxs = np.random.choice(W_.size, n_up, replace=False)
                    flat = W_.ravel()
                    for j in idxs:
                        orig    = flat[j]
                        flat[j] = orig + eps
                        setattr(self, name, flat.reshape(W_.shape))
                        pp = self._forward(xi, training=False)
                        flat[j] = orig - eps
                        setattr(self, name, flat.reshape(W_.shape))
                        pm = self._forward(xi, training=False)
                        flat[j] = orig
                        setattr(self, name, flat.reshape(W_.shape))
                        grad.ravel()[j] = (
                            np.mean((pp - yi) ** 2) - np.mean((pm - yi) ** 2)
                        ) / (2 * eps)
                    setattr(self, name,
                            opt.update(name, getattr(self, name), grad))

            mean_loss = epoch_loss / max(N, 1)
            if hasattr(bar, "set_postfix"):
                bar.set_postfix(loss=f"{mean_loss:.4f}")
            if stopper.step(mean_loss, self._snapshot()):
                if stopper.best_state is not None:
                    self._restore(stopper.best_state)
                break
        else:
            if stopper.best_state is not None:
                self._restore(stopper.best_state)
        return self

    # ------------------------------------------------------------------
    # Inference
    # ------------------------------------------------------------------

    def predict(self, X: np.ndarray) -> np.ndarray:
        """
        Parameters
        ----------
        X : np.ndarray  Shape (N, W, F).

        Returns
        -------
        np.ndarray  Shape (N, horizon).
        """
        return np.array([self._forward(xi, training=False) for xi in X])


## 6. SARIMA (inlined)


In [ ]:
import statsmodels.api as sm

def fit_sarima(series: np.ndarray, horizon: int,
               order=(1, 1, 1), seasonal_order=(1, 1, 0, 12)) -> np.ndarray:
    """
    Fit a SARIMA model and return *horizon*-step forecast.

    Falls back to a seasonal naive forecast if fitting fails.
    """
    try:
        mod = sm.tsa.statespace.SARIMAX(
            series,
            order=order,
            seasonal_order=seasonal_order,
            enforce_stationarity=False,
            enforce_invertibility=False,
        )
        res = mod.fit(disp=False, maxiter=100)
        return np.clip(res.forecast(steps=horizon), 0, None)
    except Exception:
        if len(series) >= 12:
            return np.array(
                [series[-12 + i % 12] for i in range(horizon)], dtype=float
            )
        return np.full(horizon, series.mean())


## 7. Holt-Winters (inlined)


In [ ]:
def fit_holt_winters(series: np.ndarray, horizon: int, seasonal_periods: int = 12) -> np.ndarray:
    """Additive Holt-Winters; drops season if the series is too short."""
    y = np.asarray(series, dtype=float).ravel()
    n = len(y)
    sp = min(seasonal_periods, max(2, n // 2))
    use_seas = n >= sp * 2
    kw = dict(trend="add", initialization_method="estimated")
    if use_seas:
        kw["seasonal"] = "add"
        kw["seasonal_periods"] = sp
    try:
        res = ExponentialSmoothing(y, **kw).fit(optimized=True)
        return np.clip(np.asarray(res.forecast(horizon), dtype=float), 0, None)
    except Exception:
        if n >= 12:
            return np.array([y[-12 + i % 12] for i in range(horizon)], dtype=float)
        return np.full(horizon, float(y[-1]) if n else 0.0)


def inverse_mae_ensemble(forecasts: dict[str, np.ndarray], mae: dict[str, float]) -> np.ndarray:
    """Blend model forecasts with weights 1 / MAE."""
    names = [k for k in forecasts if k in mae and np.isfinite(mae[k])]
    if not names:
        arrs = [np.asarray(v, dtype=float) for v in forecasts.values()]
        return np.mean(np.stack(arrs, axis=0), axis=0)
    inv = {k: 1.0 / (mae[k] + 1e-9) for k in names}
    tot = sum(inv.values())
    H = len(next(iter(forecasts.values())))
    out = np.zeros(H)
    for k in names:
        fc = np.asarray(forecasts[k], dtype=float).ravel()[:H]
        out += (inv[k] / tot) * fc
    return np.clip(out, 0, None)


## 8. Optional Keras CNN-LSTM

Uses TensorFlow if installed. Skip this cell if `tensorflow` is missing — NumPy models still run.


In [ ]:
HAS_TF = False
try:
    from tensorflow.keras.callbacks import EarlyStopping
    from tensorflow.keras.layers import Conv1D, Dense, Dropout, Flatten, LSTM
    from tensorflow.keras.models import Sequential
    from tensorflow.keras.optimizers import Adam

    HAS_TF = True

    def build_keras_cnn_lstm(time_step: int, n_features: int, lr: float = 1e-3, dropout: float = 0.2):
        model = Sequential()
        model.add(Conv1D(filters=64, kernel_size=2, activation="relu",
                         input_shape=(time_step, n_features)))
        model.add(Dropout(dropout))
        model.add(LSTM(50, return_sequences=True))
        model.add(Dropout(dropout))
        model.add(LSTM(50, return_sequences=False))
        model.add(Dropout(dropout))
        model.add(Flatten())
        model.add(Dense(1))
        model.compile(optimizer=Adam(learning_rate=lr), loss="mean_squared_error")
        return model
except Exception as exc:
    print("TensorFlow not available — Keras CNN-LSTM skipped:", type(exc).__name__)


## 9. Load a monthly series (pandas only)

Reads `data/Test_All.csv` if present (claim counts by process month for one part).
Otherwise builds a synthetic seasonal series so the notebook still runs.


In [ ]:
def load_monthly_claims(csv_path: Path, part: str | None = None) -> pd.Series:
    df = pd.read_csv(csv_path)
    part_col = "Part Name" if "Part Name" in df.columns else df.columns[0]
    date_col = "PROCESSING_DATE" if "PROCESSING_DATE" in df.columns else None
    if date_col is None:
        raise ValueError("CSV needs PROCESSING_DATE")
    df[date_col] = pd.to_datetime(df[date_col], errors="coerce", dayfirst=True)
    df = df.dropna(subset=[date_col])
    if part is None:
        part = str(df[part_col].value_counts().idxmax())
    sub = df[df[part_col].astype(str) == str(part)]
    monthly = (
        sub.groupby(sub[date_col].dt.to_period("M"))
        .size()
        .rename("claims")
        .sort_index()
    )
    full = pd.period_range(monthly.index.min(), monthly.index.max(), freq="M")
    monthly = monthly.reindex(full, fill_value=0).astype(float)
    monthly.index = monthly.index.to_timestamp()
    return monthly, part


def synthetic_claims(n: int = 48) -> pd.Series:
    t = np.arange(n)
    y = 40 + 0.15 * t + 8 * np.sin(2 * np.pi * t / 12) + np.random.default_rng(SEED).normal(0, 2, n)
    idx = pd.date_range("2021-01-01", periods=n, freq="MS")
    return pd.Series(np.clip(y, 0, None), index=idx, name="claims")


csv = Path("data/Test_All.csv")
if csv.is_file():
    claims, PART = load_monthly_claims(csv)
    print(f"Loaded {csv} part={PART!r}  n={len(claims)}")
else:
    claims, PART = synthetic_claims(), "synthetic"
    print("CSV not found — using synthetic series")

claims.head()


## 10. Sliding windows (written here, not imported)


In [ ]:
def make_features(y: np.ndarray) -> np.ndarray:
    """Claims plus simple lags / calendar — all computed in this notebook."""
    s = pd.Series(y)
    n = len(s)
    month = np.array([pd.Timestamp(claims.index[i]).month for i in range(n)], dtype=float)
    feat = pd.DataFrame({
        "lag1": s.shift(1),
        "lag12": s.shift(12),
        "rm3": s.rolling(3, min_periods=1).mean(),
        "month": month,
        "sin12": np.sin(2 * np.pi * np.arange(n) / 12),
        "cos12": np.cos(2 * np.pi * np.arange(n) / 12),
    }).bfill().ffill().fillna(0.0)
    return feat.to_numpy(dtype=float)


def build_windows(series_vals: np.ndarray, exog: np.ndarray, lookback: int, horizon: int):
    n = len(series_vals) - lookback - horizon + 1
    if n <= 0:
        return None, None
    X, Y = [], []
    for i in range(n):
        win_y = series_vals[i:i + lookback, None]
        win_x = exog[i:i + lookback]
        X.append(np.concatenate([win_y, win_x], axis=1))
        Y.append(series_vals[i + lookback:i + lookback + horizon])
    return np.asarray(X), np.asarray(Y)


y_raw = claims.to_numpy(dtype=float)
exog_raw = make_features(y_raw)
c_scaler = MinMaxScaler()
e_scaler = MinMaxScaler()
y_sc = c_scaler.fit_transform(y_raw[:, None]).ravel()
ex_sc = e_scaler.fit_transform(exog_raw)

W = min(LOOKBACK, max(4, len(y_raw) // 3))
X, Y = build_windows(y_sc, ex_sc, W, HORIZON)
assert X is not None and len(X) >= 3, "Series too short — need more months"
last_window = X[-1:]
n_features = X.shape[2]
print(f"windows={len(X)}  lookback={W}  features={n_features}  horizon={HORIZON}")


## 11. Run each model independently


In [ ]:
def inv_claims(arr) -> np.ndarray:
    a = np.clip(np.asarray(arr, dtype=float).ravel(), 0, 1)
    return c_scaler.inverse_transform(a[:, None]).ravel().clip(0)


forecasts = {}

# --- Holt-Winters (claims only) ---
forecasts["Holt-Winters"] = fit_holt_winters(y_raw, HORIZON)
print("Holt-Winters", forecasts["Holt-Winters"].round(2))

# --- SARIMA (claims only) ---
forecasts["SARIMA"] = fit_sarima(y_sc, HORIZON)
forecasts["SARIMA"] = inv_claims(forecasts["SARIMA"])
print("SARIMA", forecasts["SARIMA"].round(2))

# --- CNN-LSTM ---
cnn = CnnLstmForecaster(
    lookback=W, n_features=n_features, horizon=HORIZON,
    lr=1e-2, epochs=DEMO_EPOCHS, seed=SEED, dropout=0.2, early_patience=3,
)
cnn.fit(X, Y)
forecasts["CNN-LSTM"] = inv_claims(cnn.predict(last_window)[0])
print("CNN-LSTM", forecasts["CNN-LSTM"].round(2))

# --- N-BEATS ---
nb = NBeatsForecaster(
    lookback=W, n_features=n_features, horizon=HORIZON,
    lr=5e-3, epochs=DEMO_EPOCHS, seed=SEED, dropout=0.2, early_patience=3,
)
nb.fit(X, Y)
forecasts["N-BEATS"] = inv_claims(nb.predict(last_window)[0])
print("N-BEATS", forecasts["N-BEATS"].round(2))

# --- Transformer ---
tfm = TransformerForecaster(
    lookback=W, n_features=n_features, horizon=HORIZON,
    d_model=16, n_heads=2, lr=3e-3, epochs=DEMO_EPOCHS, seed=SEED,
    dropout=0.2, early_patience=3,
)
tfm.fit(X, Y)
forecasts["Transformer"] = inv_claims(tfm.predict(last_window)[0])
print("Transformer", forecasts["Transformer"].round(2))


## 12. Keras CNN-LSTM (optional, 1-step then repeated)


In [ ]:
if HAS_TF:
    X1, y1 = [], []
    for i in range(len(y_sc) - W):
        X1.append(np.concatenate([y_sc[i:i + W, None], ex_sc[i:i + W]], axis=1))
        y1.append(y_sc[i + W])
    X1, y1 = np.asarray(X1), np.asarray(y1)
    km = build_keras_cnn_lstm(W, n_features)
    km.fit(X1, y1, epochs=max(DEMO_EPOCHS, 12), verbose=0,
           callbacks=[EarlyStopping(patience=3, restore_best_weights=True)])
    hist = y_sc.copy()
    ex = ex_sc.copy()
    keras_fc = []
    for _ in range(HORIZON):
        win = np.concatenate([hist[-W:, None], ex[-W:]], axis=1)[None]
        p = float(np.clip(km.predict(win, verbose=0).ravel()[0], 0, 1))
        keras_fc.append(p)
        hist = np.append(hist, p)
        ex = np.vstack([ex, ex[-1]])
    forecasts["Keras-CNN-LSTM"] = inv_claims(keras_fc)
    print("Keras-CNN-LSTM", forecasts["Keras-CNN-LSTM"].round(2))
else:
    print("Skipped Keras-CNN-LSTM")


## 13. Ensemble + table


In [ ]:
# Toy in-sample MAE on last HORIZON actuals vs each model's first-step style score
hold = min(HORIZON, len(y_raw) // 5)
actual_tail = y_raw[-hold:]
mae = {}
for name, fc in forecasts.items():
    pred = np.asarray(fc, dtype=float).ravel()[:hold]
    if len(pred) < hold:
        continue
    # Compare last hold actuals to the mean of the forecast (demo only)
    mae[name] = float(np.mean(np.abs(actual_tail - pred[:hold])))

forecasts["Ensemble"] = inverse_mae_ensemble(
    {k: v for k, v in forecasts.items() if k != "Ensemble"}, mae
)

future_idx = pd.date_range(claims.index[-1] + pd.offsets.MonthBegin(1), periods=HORIZON, freq="MS")
table = pd.DataFrame({k: np.asarray(v, dtype=float).ravel()[:HORIZON] for k, v in forecasts.items()},
                     index=future_idx)
table.index.name = "Month"
table.round(2)
print("Demo MAE (not walk-forward):", {k: round(v, 3) for k, v in mae.items()})


## 14. Plot


In [ ]:
if HAS_MPL:
    fig, ax = plt.subplots(figsize=(10, 4.5))
    ax.plot(claims.index, y_raw, label="history", color="#1D4ED8")
    colors = {
        "Holt-Winters": "#B45309", "SARIMA": "#047857", "CNN-LSTM": "#6C63FF",
        "N-BEATS": "#FF6584", "Transformer": "#0D9488", "Keras-CNN-LSTM": "#7C3AED",
        "Ensemble": "#111827",
    }
    for name, fc in forecasts.items():
        ax.plot(future_idx, fc, label=name, linestyle="--" if name != "Ensemble" else "-",
                color=colors.get(name), linewidth=2 if name == "Ensemble" else 1.4)
    ax.set_title(f"Standalone models — {PART}")
    ax.legend(fontsize=8, ncol=2)
    ax.set_ylabel("Claims")
    fig.tight_layout()
    plt.show()
else:
    print(table.round(2))
